# 05 - Rolling LightGBM fusion
Fit purged rolling models and save the latest checkpoint plus daily stock scores.

In [ ]:
from pathlib import Path
import os
if not Path('configs/training_config.yaml').exists():
    repo=os.environ.get('ALPHAMINING_REPO_URL','https://github.com/YOUR_GITHUB_USERNAME/AlphaMining-GFlowNet-AlphaEval.git')
    if 'YOUR_GITHUB_USERNAME' in repo: raise RuntimeError('Set ALPHAMINING_REPO_URL')
    !git clone $repo
    %cd AlphaMining-GFlowNet-AlphaEval

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA status:',torch.cuda.is_available(),'CUDA runtime:',torch.version.cuda)
if torch.cuda.is_available():
    p=torch.cuda.get_device_properties(0); print('GPU model:',p.name); print('GPU memory (GB):',round(p.total_memory/1024**3,2))
else: print('GPU model: unavailable; GPU memory (GB): 0')

In [ ]:
import yaml,pandas as pd
config=yaml.safe_load(open('configs/training_config.yaml',encoding='utf-8'))
from src.model import LightGBMConfig,LightGBMFusion
evaluation=pd.read_csv('results/alpha_eval_result.csv')
selected=evaluation.loc[evaluation.dpp_selected.astype(bool),'factor'].tolist()
fusion=LightGBMFusion(LightGBMConfig(**config['lightgbm']))
prediction=fusion.fit_predict(pd.read_pickle(config['dataset']['output']),pd.read_pickle('results/alpha_factor_matrix.pkl'),selected)
display(prediction.tail(30))
loaded=LightGBMFusion.load('results/lightgbm/lgbm_model.joblib')
print('Reload successful; features:',len(loaded['features']))